# Conversion to ONNX

The image size with dependencies, the model weights file, and the runtime memory combined makes this load too slow for Sagemaker Serverless. The deployment health check is timing out. To resolve this issue, this converts the model file to ONNX for a smaller deployment size and faster startup.

In [14]:
import torch
import timm
import time
import json
import boto3
import requests
import os

In [6]:
session = boto3.Session(profile_name="animl")
s3 = session.client("s3")

In [20]:
model_name = "small-animal-classifier"
model_version = "v1.0"
asset_name = "eva02-20260630-llrd.best.e02-s053514.stripped.pt"
model_zoo_bucket = "animl-model-zoo"
local_model_path = "./model-weights"

## Store Source of Truth Weights in Animl Model Zoo

This ensures that Animl always retains a copy of the weights used to export the model to the deployed ONNX model. This only needs to be run once to save the `.pt` file to our bucket. It can also be manually copied over.

Model release: [small-animal-classifier](https://github.com/agentmorris/small-animal-classifier/releases/tag/v1.0).

In [16]:
# Download model weights from GitHub
model_file = requests.get(f"https://github.com/agentmorris/small-animal-classifier/releases/download/{model_version}/{asset_name}")
print(f"Downloaded model from github.com/agentmorris/small-animal-classifier/releases/download/{release_version}/{asset_name}")
model_file.raise_for_status()

with open(f"/tmp/{asset_name}", "wb") as f:
    f.write(model_file.content)

# Upload to Model Zoo
s3.upload_file(f"/tmp/{asset_name}", model_zoo_bucket, f"{model_name}/{asset_name}")
print(f"Uploaded {asset_name} to s3://{model_zoo_bucket}/{model_name}/{asset_name}")

# (Optional) Cleanup temp file
os.remove(f"/tmp/{asset_name}")
print(f"Removed temp file from /tmp/{asset_name}")

Downloaded model from github.com/agentmorris/small-animal-classifier/releases/download/v1.0/eva02-20260630-llrd.best.e02-s053514.stripped.pt
Uploaded eva02-20260630-llrd.best.e02-s053514.stripped.pt to s3://animl-model-zoo/small-animal-classifier/eva02-20260630-llrd.best.e02-s053514.stripped.pt
Removed temp file from /tmp/eva02-20260630-llrd.best.e02-s053514.stripped.pt


## Load the Checkpoint From the Animl Model Zoo

Downloading from our saved copy ensures that the model weights are exactly the same weights used to export to ONNX for the deployed model.

In [21]:
s3.download_file(model_zoo_bucket, f"{model_name}/{asset_name}", f"{local_model_path}/{asset_name}")

ck = torch.load(f"{local_model_path}/{asset_name}", map_location="cpu", weights_only=False)

## Rebuild the Model

In [9]:
model = timm.create_model(
    ck["model_name"],
    pretrained=False,
    num_classes=ck["num_classes"]
)
model.load_state_dict(ck["state_dict"])
model.eval()

print(f"Model: {ck["model_name"]}, classes: {ck["num_classes"]}, input: {ck["img_size"]}x{ck["img_size"]}")

Model: eva02_large_patch14_448.mim_m38m_ft_in22k_in1k, classes: 29, input: 448x448


## Export to ONNX

In [18]:
img_size = ck["img_size"]
dummy_input = torch.randn(1, 3, img_size, img_size)

torch.onnx.export(
    model,
    dummy_input,
    "model-weights/small-animal-classifier.onnx",
    input_names=["input"],
    output_names=["logit"],
    opset_version=17,
    dynamo=True
)

print("Model export complete")

metadata = {
    "model_name": ck["model_name"],
    "num_classes": ck["num_classes"],
    "img_size": ck["img_size"],
    "classes": ck["classes"],
    "norm_mean": ck["norm_mean"],
    "norm_std": ck["norm_std"],
    "banner_crop": ck["banner_crop"]
}

with open("./model-weights/small-animal-classifier-metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Model metadata saved")

W0817 10:04:07.266000 73792 torch/onnx/_internal/exporter/_schemas.py:454] Missing annotation for parameter 'input' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0817 10:04:07.267000 73792 torch/onnx/_internal/exporter/_schemas.py:454] Missing annotation for parameter 'rois' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0817 10:04:07.268000 73792 torch/onnx/_internal/exporter/_schemas.py:454] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.
W0817 10:04:07.268000 73792 torch/onnx/_internal/exporter/_schemas.py:454] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.


[torch.onnx] Obtain model graph for `Eva([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `Eva([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...


Conversion to opset < 18 is not supported.


[torch.onnx] Translate the graph into ONNX... ✅
Model export complete
Model metadata saved


## Quick Sanity Check of ONNX Model

In [14]:
import onnxruntime as ort
import numpy as np

In [15]:
session = ort.InferenceSession("model-weights/small-animal-classifier.onnx")

2026-08-17 09:44:27.796 Python[73792:28139833] 2026-08-17 09:44:27.795197 [W:onnxruntime:, graph.cc:5566 CleanUnusedInitializersAndNodeArgs] Removing initializer 'val_6914'. It is not used by any node and should be removed from the model.
2026-08-17 09:44:27.796 Python[73792:28139833] 2026-08-17 09:44:27.796434 [W:onnxruntime:, graph.cc:5566 CleanUnusedInitializersAndNodeArgs] Removing initializer 'val_6899'. It is not used by any node and should be removed from the model.
2026-08-17 09:44:27.796 Python[73792:28139833] 2026-08-17 09:44:27.796449 [W:onnxruntime:, graph.cc:5566 CleanUnusedInitializersAndNodeArgs] Removing initializer 'val_6877'. It is not used by any node and should be removed from the model.
2026-08-17 09:44:27.796 Python[73792:28139833] 2026-08-17 09:44:27.796458 [W:onnxruntime:, graph.cc:5566 CleanUnusedInitializersAndNodeArgs] Removing initializer 'val_6843'. It is not used by any node and should be removed from the model.
2026-08-17 09:44:27.796 Python[73792:2813983

In [16]:
test_input = torch.randn(1, 3, img_size, img_size)

with torch.no_grad():
    pt_logits = model(test_input).numpy()

ort_logits = session.run(None, {"input": test_input.numpy()})[0]

max_diff = np.abs(pt_logits - ort_logits).max()
print(f"Max absolute difference: {max_diff:.8f}")
print(f"Match {"yes" if max_diff < 1e-4 else "no"}")

Max absolute difference: 0.00000536
Match yes


## Upload to Animl Model Zoo

This makes it available for other developers and for the deployed Sagemaker endpoint to use.

In [25]:
s3.upload_file(f"{local_model_path}/{model_name}.onnx", model_zoo_bucket, f"{model_name}/{model_name}.onnx")
print(f"Uploaded {local_model_path}/{model_name}.onnx to {model_zoo_bucket}/{model_name}/{model_name}.onnx")

s3.upload_file(f"{local_model_path}/{model_name}.onnx.data", model_zoo_bucket, f"{model_name}/{model_name}.onnx.data")
print(f"Uploaded {local_model_path}/{model_name}.onnx.data to {model_zoo_bucket}/{model_name}/{model_name}.onnx.data")

s3.upload_file(f"{local_model_path}/{model_name}-metadata.json", model_zoo_bucket, f"{model_name}/{model_name}-metadata.json")
print(f"Uploaded {local_model_path}/{model_name}-metadata.json to {model_zoo_bucket}/{model_name}/{model_name}-metadata.json")

Uploaded ./model-weights/small-animal-classifier.onnx to animl-model-zoo/small-animal-classifier/small-animal-classifier.onnx
Uploaded ./model-weights/small-animal-classifier.onnx.data to animl-model-zoo/small-animal-classifier/small-animal-classifier.onnx.data
Uploaded ./model-weights/small-animal-classifier-metadata.json to animl-model-zoo/small-animal-classifier/small-animal-classifier-metadata.json
